# 05 - Feature engineering experiment

Tuning in the previous notebook barely moved the score, which suggested the limit is the information in the raw signals rather than the model. A natural next question: do physically meaningful combinations of the existing signals add anything the booster can't already recover on its own?

I test three, each computed per row from existing columns so there's no leakage:

- `i_s` = sqrt(i_d^2 + i_q^2), the stator current magnitude
- `u_s` = sqrt(u_d^2 + u_q^2), the voltage magnitude
- `p_el` = u_d*i_d + u_q*i_q, an electrical power proxy in the d/q frame

The decision is made on grouped cross-validation, not on the held-out test set. Using the test set to choose features would quietly leak it into model selection.

In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold, cross_val_score

from src.data_prep import load_raw, clean, split_by_profile
from src.model import build_model
from src.evaluate import regression_metrics

df = clean(load_raw())
X_train, X_test, y_train, y_test, groups_train = split_by_profile(df)
X_sel, y_sel, g_sel = X_train.iloc[::5], y_train.iloc[::5], groups_train[::5]


def engineer(X):
    X = X.copy()
    X["i_s"] = np.sqrt(X["i_d"] ** 2 + X["i_q"] ** 2)
    X["u_s"] = np.sqrt(X["u_d"] ** 2 + X["u_q"] ** 2)
    X["p_el"] = X["u_d"] * X["i_d"] + X["u_q"] * X["i_q"]
    return X

## Grouped CV: 7 features vs 10

In [2]:
gkf = GroupKFold(n_splits=4)
rows = []
for label, transform in [("baseline_7", lambda d: d), ("engineered_10", engineer)]:
    cv = -cross_val_score(build_model(), transform(X_sel), y_sel, groups=g_sel,
                          cv=gkf, scoring="neg_root_mean_squared_error")
    rows.append({"features": label, "cv_rmse": cv.mean(), "cv_std": cv.std()})
pd.DataFrame(rows).set_index("features").round(3)

,cv_rmse,cv_std
features,,
baseline_7,12.207,1.337
engineered_10,12.256,1.265


The two sets are tied to within a small fraction of an RMSE point, and the difference is far inside the fold-to-fold standard deviation of about 1.3. By the criterion that decides, grouped CV, the engineered features add nothing.

## Holdout, for reference only

In [3]:
rows = []
for label, transform in [("baseline_7", lambda d: d), ("engineered_10", engineer)]:
    m = build_model().fit(transform(X_train), y_train)
    te = regression_metrics(y_test, m.predict(transform(X_test)))
    rows.append({"features": label, "test_rmse": te["rmse"], "test_r2": te["r2"]})
pd.DataFrame(rows).set_index("features").round(3)

,test_rmse,test_r2
features,,
baseline_7,10.843,0.671
engineered_10,10.559,0.688


## Decision

On the holdout the engineered set looks slightly better, but that is exactly the number I'm not allowed to optimise against, picking features by the test score would leak it into selection and inflate my sense of how well the model generalises. The honest criterion, grouped CV, shows no gain.

The interpretation is straightforward: `i_s`, `u_s` and `p_el` are deterministic functions of inputs the model already has, and a gradient booster can approximate those shapes by itself, so handing them over explicitly doesn't add information. I therefore keep the seven raw features. The exercise is still worth having done, it confirms where the ceiling comes from (the signals, not the representation) and it shows the feature choice was tested rather than assumed.